<a href="https://colab.research.google.com/github/WeridFire/QuantumComputing/blob/main/assignment1/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 - State‑Vector Quantum Computer Simulator
You will build a minimal state‑vector simulator and use it to simulate a 2‑qubit Bell circuit. A state‑vector simulator represents the quantum state as a complex vector and applies gates as matrices.

> Convention: We use computational basis ordering |q0 q1> with q0 as the most significant qubit (MSB). The basis is |00>,|01>,|10>,|11> → indices [0,1,2,3].

## Task 1.1 - State‑vector representation
Getting the basis and normalization right prevents subtle bugs later (wrong amplitudes, swapped qubits). This is the foundation for applying gates and interpreting measurements.

> What the code does: Creates and normalizes a 4‑dimensional complex vector for a 2‑qubit system, initializing it to |00> or |0>.

In [1]:
import numpy as np

DTYPE = np.complex128

def ket00():
    """Return |00> as a length‑4 complex column vector."""
    v = np.zeros(4, dtype=DTYPE)
    v[0] = 1.0
    return v

def normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("Zero vector cannot be normalized.")
    return v / n

# Example
psi = ket00()
assert np.isclose(np.linalg.norm(psi), 1.0)

> What to include. A short sentence describing your basis ordering and how you map bitstrings to indices.

### Our Solution

We adopt the standard computational basis ordering $\vert{}q_0 q_1\rangle$, where $q_0$ is the most significant qubit (MSB) and $q_1$ is the least significant qubit (LSB). Each two-qubit basis state directly maps to its integer array index via the standard binary representation.

## Task 1.2 - Quantum gate implementation

Gates are linear operators. Building them once and applying correctly to the target qubit makes later circuits easy to reason about and unit‑test.

> What the code does: Defines Hadamard and CNOT matrices and utilities to apply them in the 2‑qubit basis.

In [ ]:
# 1‑qubit and 2‑qubit gate definitions (matrix form)
I2 = np.array([[1, 0], [0, 1]], dtype=DTYPE)
H  = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=DTYPE)

# CNOT with control = q0 (MSB), target = q1 (LSB) in basis |q0 q1>
CNOT_01 = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,0,1],
    [0,0,1,0]
], dtype=DTYPE)

# Expand a 1‑qubit unitary U to a 2‑qubit operator acting on target qubit t in |q0 q1> order
# t = 0 applies to q0 (MSB); t = 1 applies to q1 (LSB)

def expand_1q(U: np.ndarray, t: int) -> np.ndarray:
    if t == 0:   # act on q0 (MSB)
        return np.kron(U, I2)
    elif t == 1: # act on q1 (LSB)
        return np.kron(I2, U)
    else:
        raise ValueError("target index must be 0 or 1")

# Apply a gate matrix to a state vector

def apply(U: np.ndarray, psi: np.ndarray) -> np.ndarray:
    return U @ psi

# Quick unit tests
assert np.allclose(H.conj().T @ H, I2)
assert np.allclose(CNOT_01.conj().T @ CNOT_01, np.eye(4))

> What to include. Briefly justify your control/target convention for CNOT and how it matches the basis ordering.

### Our Solution


##Task 1.3 - Circuit simulation: Bell state

The Bell state is the simplest non‑separable state. Simulating it shows how interference from H, combined with a single controlled operation, produces entanglement.

> What the code does: Applies H on q0 then CNOT(q0→q1) to produce |Φ+> = (|00>+|11>)/√2.

In [ ]:
def bell_state() -> np.ndarray:
    psi = ket00()
    # H on q0 (MSB), then CNOT(control=q0, target=q1)
    psi = apply(expand_1q(H, t=0), psi)
    psi = apply(CNOT_01, psi)
    return normalize(psi)

psi_bell = bell_state()
print("Bell state vector:", np.round(psi_bell, 6))
# Expected amplitudes (up to a global phase): [1/√2, 0, 0, 1/√2]

> What to include. Print the final state and confirm it matches ( |00> + |11> ) / √2 up to global phase.

### Our Solution


##Task 1.4 - Output, verification, and circle‑plot comparison

Bridge amplitudes -> measurement outcomes, and verify your simulator against expectations. Visualization connects algebra to intuitive structure.

> What the code does: Samples bitstrings from the state, plots a histogram, and compare to earlier work.

In [ ]:
import matplotlib.pyplot as plt

BITSTR = ["00","01","10","11"]

def sample_bitstrings(psi: np.ndarray, shots: int = 10_000, seed: int = 7):
    rng = np.random.default_rng(seed)
    probs = np.abs(psi)**2
    idx = rng.choice(4, size=shots, p=probs)
    return idx

# Sample from Bell state
shots = 10_000
idx = sample_bitstrings(psi_bell, shots=shots)

# Make histogram
counts = {b: int(np.sum(idx == i)) for i, b in enumerate(BITSTR)}
print(counts)
plt.bar(counts.keys(), counts.values())
plt.xlabel("bitstring"); plt.ylabel("counts"); plt.title("Bell state measurement histogram")
plt.show()

> What to include:
   - Circle‑notation visualization. Use your lecture function to visualize (in circle notation, see here, for instance) psi_bell

  - Compare and reflect. Do the counts match expectations within sampling error? How does the circle plot show two dominant components |00> and |11> and their relative phase?

### Our Solution
